# CNeuroMod QA — tSNR brain maps

Volumetric montages of average temporal-SNR maps (MNI space): one panel per subject, one per dataset, and one grand average pooling every subject across every dataset — read directly from the `tsnr` derivative of `source_data/cneuromod.all/{dataset}/`. Figures are written to `output_data/figures/tsnr_maps/`.

tSNR is volumetric and QA cares about signal dropout in ventral/orbitofrontal, temporal and subcortical regions, so we render faithful volumetric slices (nilearn) rather than a cortical surface, which would discard subcortex/cerebellum.

In [1]:
import os
from pathlib import Path

import nibabel as nib
import numpy as np
from nilearn import plotting

# Paths are provided by `invoke run-notebooks` as environment variables.
# Figures go in output_data/figures/{FIG_NAME}/ (also the notebook's "already
# ran" sentinel); the avgtsnr maps this notebook reads are never persisted —
# they live only in the source `tsnr` derivative, fetched by `invoke fetch`.
FIG_NAME = "tsnr_maps"
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data"))
SOURCE_DIR = Path(os.environ.get("SOURCE_DATA_DIR", "../source_data")) / "cneuromod.all"
FIG_DIR = OUTPUT_DIR / "figures" / FIG_NAME
FIG_DIR.mkdir(parents=True, exist_ok=True)

# The upstream per-subject average tSNR map, MNI space (see analysis/tsnr_maps.py).
SPACE = "MNI152NLin2009cAsym"
SUBJECT_AVG_GLOB = f"sub-*/sub-*_space-{SPACE}_stat-avgtsnr_statmap.nii.gz"

# Shared display settings so panels are visually comparable.
CMAP = "inferno"
DISPLAY_MODE = "z"       # axial montage — shows ventral/subcortical dropout
N_CUTS = 8
DPI = 120

In [2]:
from nilearn.image import resample_to_img
from nilearn.plotting import find_cut_slices


def discover_datasets():
    """Dataset names under SOURCE_DIR whose tsnr derivative has >=1 avgtsnr map."""
    datasets = []
    if SOURCE_DIR.is_dir():
        for tsnr_dir in sorted(SOURCE_DIR.glob("*/tsnr")):
            maps = [p for p in tsnr_dir.glob(SUBJECT_AVG_GLOB) if p.is_file()]
            if maps:
                datasets.append(tsnr_dir.parent.name)
    return datasets


def subject_maps(dataset):
    """Sorted per-subject avgtsnr map paths for one dataset, read from source_data."""
    tsnr_dir = SOURCE_DIR / dataset / "tsnr"
    return sorted(p for p in tsnr_dir.glob(SUBJECT_AVG_GLOB) if p.is_file())


def average_image(paths):
    """Mean tSNR image over a list of subject maps, computed in memory (nothing written).

    Maps may sit on slightly different grids, so each is resampled to the first
    map's grid before averaging. NaNs are ignored voxelwise so a subject missing
    coverage never blanks a voxel for everyone.
    """
    reference = nib.load(str(paths[0]))
    stack = []
    for path in paths:
        image = nib.load(str(path))
        if image.shape != reference.shape or not np.allclose(image.affine, reference.affine):
            image = resample_to_img(image, reference, copy_header=True)
        stack.append(np.asarray(image.dataobj, dtype=np.float32))
    mean = np.nanmean(np.stack(stack, axis=-1), axis=-1)
    return nib.Nifti1Image(mean, reference.affine, reference.header)


def dataset_average_image(dataset):
    """Mean tSNR image over one dataset's subject maps."""
    return average_image(subject_maps(dataset))


def robust_vmax(images):
    """98th percentile of positive tSNR across images — a shared, outlier-robust ceiling."""
    values = []
    for image in images:
        data = np.asarray(image.dataobj, dtype=np.float32)
        data = data[np.isfinite(data) & (data > 0)]
        if data.size:
            values.append(np.percentile(data, 98))
    return float(np.median(values)) if values else None


datasets = discover_datasets()
print(f"found avgtsnr maps for: {datasets}")
if not datasets:
    print("No tSNR maps found — run `invoke fetch` first (needs data access).")
dataset_images = {dataset: dataset_average_image(dataset) for dataset in datasets}

# Grand average: every subject from every (light-v1) dataset pooled into one
# mean map, each subject weighted equally regardless of how many subjects its
# dataset contributes — a single cross-dataset "house average" tSNR panel.
all_subject_paths = [path for dataset in datasets for path in subject_maps(dataset)]
grand_average_image = average_image(all_subject_paths) if all_subject_paths else None

vmax_images = list(dataset_images.values())
if grand_average_image is not None:
    vmax_images.append(grand_average_image)
VMAX = robust_vmax(vmax_images)
print(f"shared vmax = {VMAX}")

# Cut coordinates are chosen once per dataset from its average map, then reused
# for every subject panel in that dataset — so slices are picked from the
# strongest, least noisy signal (the average) rather than autoselected
# independently per subject, keeping panels anatomically comparable.
dataset_cut_coords = {
    dataset: find_cut_slices(image, direction=DISPLAY_MODE, n_cuts=N_CUTS)
    for dataset, image in dataset_images.items()
}
grand_average_cut_coords = (
    find_cut_slices(grand_average_image, direction=DISPLAY_MODE, n_cuts=N_CUTS)
    if grand_average_image is not None else None
)


found avgtsnr maps for: ['floc', 'retinotopy', 'things']


shared vmax = 52.897544860839844


In [3]:
from nilearn.datasets import fetch_icbm152_2009

THRESHOLD = 30
COVERAGE_CMAP = "viridis"
NILEARN_DIR = Path(os.environ.get("SOURCE_DATA_DIR", "../source_data")) / "nilearn"


def binary_coverage_image(paths, threshold, brain_mask_path):
    """Fraction of subjects with tsnr above `threshold` at each voxel, in-brain only.

    Same load/resample loop as `average_image`, but each subject's map is
    thresholded to a 0/1 mask before averaging — turning "how good is signal"
    into "how many subjects had usable signal at all", a direct coverage/
    dropout QA signal. The averaged fraction is then zeroed outside the MNI
    whole-brain mask: a low tSNR threshold alone is not enough to exclude
    background/skull voxels, whose thermal-noise tSNR routinely clears it too.
    """
    reference = nib.load(str(paths[0]))
    stack = []
    for path in paths:
        image = nib.load(str(path))
        if image.shape != reference.shape or not np.allclose(image.affine, reference.affine):
            image = resample_to_img(image, reference, copy_header=True)
        data = np.asarray(image.dataobj, dtype=np.float32)
        stack.append((data > threshold).astype(np.float32))
    mean = np.nanmean(np.stack(stack, axis=-1), axis=-1)

    mask_image = resample_to_img(
        nib.load(str(brain_mask_path)), reference,
        interpolation="nearest", copy_header=True,
    )
    in_brain = np.asarray(mask_image.dataobj) > 0
    mean = np.where(in_brain, mean, 0.0)
    return nib.Nifti1Image(mean, reference.affine, reference.header)


# The MNI152 T1 template (anatomical background) and whole-brain mask (to
# exclude background/skull voxels from coverage, see binary_coverage_image)
# are used below. `invoke fetch` caches them under NILEARN_DIR (see
# analysis/mni152.py, whose fetch_mni152_templates wraps the same call);
# nilearn checks the cache before downloading, so this is a cheap no-op when
# already fetched. If unavailable, warn and skip the coverage panels rather
# than raising, matching this notebook's tolerant style.
try:
    mni_templates = fetch_icbm152_2009(data_dir=str(NILEARN_DIR))
    mni_t1_path = Path(mni_templates["t1"])
    mni_mask_path = Path(mni_templates["mask"])
except Exception as error:
    print(f"\u26a0\ufe0f  MNI152 template unavailable ({error}) \u2014 run `invoke fetch` first. "
          "Skipping tSNR coverage panels.")
    mni_t1_path = None
    mni_mask_path = None

if mni_t1_path is not None:
    dataset_coverage_images = {
        dataset: binary_coverage_image(subject_maps(dataset), THRESHOLD, mni_mask_path)
        for dataset in datasets
    }
    grand_average_coverage_image = (
        binary_coverage_image(all_subject_paths, THRESHOLD, mni_mask_path)
        if all_subject_paths else None
    )


[fetch_icbm152_2009] Dataset directory found: /home/pbellec/git/cneuromod.all.qa_figures/source_data/nilearn/icbm152_2009


In [4]:
def plot_coverage(stat_map, title, out_path, cut_coords, bg_img):
    """Axial montage of a 0-1 subject-coverage map on an MNI152 anatomical background."""
    display = plotting.plot_stat_map(
        stat_map, bg_img=bg_img, display_mode=DISPLAY_MODE, cut_coords=cut_coords,
        cmap=COVERAGE_CMAP, vmin=0, vmax=1, threshold=0.01, colorbar=True,
        black_bg=False, title=title, symmetric_cbar=False,
    )
    display.savefig(str(out_path), dpi=DPI)
    display.close()


# One coverage montage per dataset (fraction of subjects with tsnr > THRESHOLD).
if mni_t1_path is not None:
    for dataset, image in dataset_coverage_images.items():
        plot_coverage(
            image, f"{dataset} \u2014 tSNR coverage (>{THRESHOLD})",
            FIG_DIR / f"{dataset}_coverage.png", dataset_cut_coords[dataset],
            bg_img=str(mni_t1_path),
        )

    # Grand-average coverage montage, pooling every subject across every dataset.
    if grand_average_coverage_image is not None:
        plot_coverage(
            grand_average_coverage_image, f"all datasets \u2014 tSNR coverage (>{THRESHOLD})",
            FIG_DIR / "all_datasets_coverage.png", grand_average_cut_coords,
            bg_img=str(mni_t1_path),
        )


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


In [5]:
def plot_tsnr(stat_map, title, out_path, cut_coords, vmax=None):
    """Axial montage of one tSNR map (path or in-memory image) on the MNI template."""
    display = plotting.plot_stat_map(
        stat_map, display_mode=DISPLAY_MODE, cut_coords=cut_coords,
        cmap=CMAP, vmax=vmax, colorbar=True, black_bg=True,
        title=title, symmetric_cbar=False,
    )
    display.savefig(str(out_path), dpi=DPI)
    display.close()


# One montage per dataset average (computed in memory, nothing written to disk).
for dataset, image in dataset_images.items():
    plot_tsnr(image, f"{dataset} — average tSNR",
              FIG_DIR / f"{dataset}_avgtsnr.png", dataset_cut_coords[dataset], vmax=VMAX)

# Grand average montage, pooling every subject across every dataset.
if grand_average_image is not None:
    plot_tsnr(grand_average_image, "all datasets — average tSNR",
              FIG_DIR / "all_datasets_avgtsnr.png", grand_average_cut_coords, vmax=VMAX)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


In [6]:
# One montage per subject, per dataset — read straight from source_data, sliced
# at the same cut coordinates as that dataset's average for comparability.
for dataset in datasets:
    for path in subject_maps(dataset):
        subject = path.name.split("_", 1)[0]  # e.g. sub-01
        plot_tsnr(str(path), f"{dataset} — {subject} tSNR",
                  FIG_DIR / f"{dataset}_{subject}_avgtsnr.png",
                  dataset_cut_coords[dataset], vmax=VMAX)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)
